# Used Car Price Prediction

**Regression Project 53**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict the resale price of used cars based on features like brand, age, kilometers driven,
fuel type, and transmission. The used-car market is one of the most practical applications
of regression in everyday life.

This project uses the CarDekho dataset, which contains real listing data from the Indian
used-car market.

## 3. Learning Objectives

1. Build a regression model for real-world vehicle pricing
2. Handle high-cardinality categorical features (car names/brands)
3. Engineer depreciation and age-based features
4. Understand how mileage, fuel type, and transmission affect resale value
5. Compare LazyPredict, FLAML, and tuned gradient boosting models
6. Evaluate with R², RMSE, MAE and residual analysis

## 4. Problem Statement

> Given used-car listing attributes, predict the **selling price** (`Selling_Price` in lakhs).

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Car buyers | Avoid overpaying for used vehicles |
| Car sellers | Set competitive asking prices |
| Dealerships | Inventory valuation and margin optimization |
| Auto-lending | Collateral assessment for car loans |
| ML learners | Real-world regression with mixed feature types |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `nehalbirla/vehicle-dataset-from-cardekho` |
| Records | ~301 |
| Features | 8 (Car_Name, Year, Present_Price, Kms_Driven, Fuel_Type, Seller_Type, Transmission, Owner) |
| Target | `Selling_Price` (continuous, in lakhs INR) |
| Key insight | Car age and present price are strongest predictors |

## 7. Dataset Source and License Notes

- **Kaggle**: [nehalbirla/vehicle-dataset-from-cardekho](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho)
- Real listing data from CarDekho.com
- License: CC0 / Public Domain
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'nehalbirla/vehicle-dataset-from-cardekho'
TARGET = 'Selling_Price'
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 90
MAX_ROWS = 50000

## 11. Dataset Download or Loading

Download the CarDekho used-car dataset via `kagglehub`.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho'
    )
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV files found in {path}'
# Pick the main car data file
car_files = [f for f in csv_files if 'car' in os.path.basename(f).lower()]
chosen = car_files[0] if car_files else sorted(csv_files, key=os.path.getsize, reverse=True)[0]
df_raw = pd.read_csv(chosen)
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 12. Data Validation Checks

Verify target, missing values, duplicates, and column types.

In [ ]:
assert TARGET in df_raw.columns, f"Target '{TARGET}' not found in {list(df_raw.columns)}"
print(f'Missing values:\n{df_raw.isnull().sum()}\n')
print(f'Duplicated rows: {df_raw.duplicated().sum()}')
df = df_raw.copy()
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Shape after cleaning: {df.shape}')
df.dtypes

## 13. Exploratory Data Analysis

Small dataset (301 rows) — we can inspect every feature thoroughly.

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, ['Year', 'Present_Price', 'Kms_Driven', TARGET]):
    if col in df.columns:
        df[col].hist(bins=25, ax=ax, alpha=0.7, color='steelblue')
        ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
cat_feats = ['Fuel_Type', 'Seller_Type', 'Transmission']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cat_feats):
    if col in df.columns:
        df.groupby(col)[TARGET].mean().plot.bar(ax=ax, color='coral')
        ax.set_title(f'Mean {TARGET} by {col}')
        ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
num_cols_eda = df.select_dtypes(include=[np.number]).columns.tolist()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[num_cols_eda].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout(); plt.show()

## 14. Target Analysis

Used car prices are right-skewed — most cars are budget/mid-range, with a few premium vehicles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=30, color='coral', alpha=0.7)
axes[0].set_title(f'{TARGET} Distribution')
axes[1].hist(np.log1p(df[TARGET]), bins=30, color='seagreen', alpha=0.7)
axes[1].set_title(f'log({TARGET}) Distribution')
plt.tight_layout(); plt.show()
print(f'Mean: {df[TARGET].mean():.2f} lakhs')
print(f'Median: {df[TARGET].median():.2f} lakhs')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split. With only ~301 rows, each split is small.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- **Numeric** (Year, Present_Price, Kms_Driven, Owner): Impute median + StandardScaler
- **Categorical** (Car_Name, Fuel_Type, Seller_Type, Transmission): Impute mode + OneHotEncoder
- `Car_Name` has high cardinality — we extract brand as a cleaner categorical

In [ ]:
# Extract brand from car name (first word)
for d in [X_train, X_val, X_test]:
    if 'Car_Name' in d.columns:
        d['brand'] = d['Car_Name'].str.split().str[0].str.lower()
        d.drop(columns=['Car_Name'], inplace=True)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')
print(f'Brands: {X_train["brand"].nunique() if "brand" in X_train.columns else 0}')

## 17. Feature Engineering

- **Car age**: 2024 minus manufacturing year
- **Depreciation ratio**: Selling_Price / Present_Price (not available as feature, but we use Present_Price as proxy)
- **Kms per year**: kilometers driven divided by car age (usage intensity)

In [ ]:
def engineer_features(d):
    d = d.copy()
    if 'Year' in d.columns:
        d['car_age'] = 2024 - d['Year']
    if 'Kms_Driven' in d.columns and 'Year' in d.columns:
        age = (2024 - d['Year']).replace(0, 1)
        d['kms_per_year'] = d['Kms_Driven'] / age
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
# Rebuild preprocessor
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')

## 18. Baseline Model

Three baselines: DummyRegressor, LinearRegression, RandomForest.

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE={r['RMSE']:.2f}, MAE={r['MAE']:.2f}")

## 19. LazyPredict Benchmark

Quick comparison of ~30 regression models.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML with 90-second budget for model selection and tuning.

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE={results['FLAML']['RMSE']:.2f}")

## 21. Top 3 Model Selection

Based on LazyPredict/FLAML results, we select three strong models.

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=200, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

Train on training set, evaluate on held-out test set.

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE={final[name]['RMSE']:.2f}, MAE={final[name]['MAE']:.2f}")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.5, s=20)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.5, s=20)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

Examining where the best model makes the largest errors.

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: {abs_errors.mean():.2f} lakhs')
print(f'Median absolute error: {np.median(abs_errors):.2f} lakhs')
print(f'90th percentile error: {np.percentile(abs_errors, 90):.2f} lakhs')
print(f'Max error: {abs_errors.max():.2f} lakhs')

## 24. Interpretation and Business Insight

- **Present price** (showroom price) is the strongest predictor — resale value scales with original cost
- **Car age** is the second driver — vehicles depreciate significantly in the first 3–5 years
- **Fuel type** matters: diesel cars retain value better due to running-cost advantages
- **Transmission**: automatic cars command a premium in resale
- **Kms driven** shows diminishing depreciation — after 50K km, additional km matter less
- **Owner count** negatively impacts price — each additional owner reduces trust

## 25. Limitations

1. Very small dataset (301 rows) — risk of overfitting
2. Limited to Indian market — not transferable to other countries
3. No vehicle condition, accident history, or service records
4. Car_Name has high cardinality; brand extraction loses model-level info
5. No regional pricing variation within India

## 26. How to Improve This Project

1. Use a larger dataset with more car listings
2. Include additional features (mileage, engine CC, max power)
3. Target encoding for brand instead of one-hot
4. Cross-validation for more robust evaluation on small data
5. Ensemble stacking of top models

## 27. Production Considerations

- Real-time pricing API for used-car marketplaces
- Model retraining with fresh listing data quarterly
- Regional price adjustments
- Confidence intervals for each estimate
- A/B testing pricing suggestions vs human appraisals

## 28. Common Mistakes

1. Keeping `Car_Name` as-is without extracting brand
2. Not creating `car_age` from `Year`
3. Including `Present_Price` without understanding it may partially leak
4. Overfitting on 301 rows with complex models
5. Not checking for outlier prices (premium luxury cars)

## 29. Mini Challenge / Exercises

1. Try log-transforming the target and compare R²
2. Build separate models for petrol vs diesel cars
3. Use 10-fold cross-validation and report mean ± std
4. Drop `Present_Price` and see how much R² drops
5. Try target encoding for `brand` instead of OHE

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict used car selling price |
| Dataset | CarDekho (~301 records) |
| Difficulty | Easy–Medium |
| Key insight | Present price + car age dominate prediction |
| Best approach | Gradient boosting with engineered features |
| Primary metric | R² |
| Baseline comparison | Tree models outperform linear regression significantly |